<!--nav--> [🗺 Learning path](README.md) · **49/49** · ◀ [VLM Serving: The Token Explosion](./VLM_Serving_Token_Explosion.ipynb) · [🏁 back to the map](README.md)

# VLM Optimization: What Actually Works, and What It Costs You

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/VLM_Optimization_Techniques.ipynb)

[VLM Serving: The Token Explosion](./VLM_Serving_Token_Explosion.ipynb) established the problem: one image is 500–7,000
tokens, arriving through a four-phase pipeline. This notebook is the toolbox — and, in the spirit of
[The Optimization Stack](./The_Optimization_Stack.ipynb), it is honest about which tools **fight each other**.

The VLM-specific techniques, roughly by leverage:

| Technique | Cuts | Risk | Part |
|---|---|---|---|
| **Resolution capping** | tokens, everywhere | task-dependent | 1 |
| **Image / embedding caching** | preprocess + ViT, entirely | none | 2 |
| **Multimodal prefix caching** | LLM prefill for repeated images | none | 2 |
| **Visual token pruning** | prefill + KV | real, and subtle | 3 |
| **Adaptive resolution routing** | tokens on easy requests | routing errors | 4 |
| **Encoder/LLM disaggregation** | interference between phases | complexity | 5 |
| **Selective quantization** | weight bytes | ViT is more fragile than the LLM | 6 |
| **Video frame sampling** | everything, dramatically | temporal detail | 7 |

**Runs on:** any CPU — all simulation and modeling.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Resolution capping is the master dial

[VLM Serving: The Token Explosion](./VLM_Serving_Token_Explosion.ipynb) Part 7 showed the span: an uncapped 4K upload can cost 10× a capped one. The nuance is
that **the right cap is task-dependent**, and getting it wrong is invisible in your metrics — the
model just quietly gets worse at reading things.

Here is the relationship you actually care about, and it is *not* linear: for most tasks quality is
a saturating function of resolution, and the saturation point differs enormously by task.

In [ ]:
import math, random, statistics

# Quality-vs-resolution profiles. These encode the SHAPE of the trade-off (saturating, with
# task-dependent knees); the absolute numbers are illustrative, not benchmark results.
TASKS = {
    "scene description":  dict(knee_px=200_000,  floor=0.72, ceiling=0.94),
    "object counting":    dict(knee_px=600_000,  floor=0.45, ceiling=0.90),
    "chart reading":      dict(knee_px=1_500_000, floor=0.20, ceiling=0.92),
    "document OCR":       dict(knee_px=3_000_000, floor=0.05, ceiling=0.95),
}

def quality(task, pixels):
    t = TASKS[task]
    # saturating curve: rises steeply until the knee, then flattens
    x = pixels / t["knee_px"]
    return t["floor"] + (t["ceiling"] - t["floor"]) * (x / (1 + x))

def tokens_for(pixels, merge=28 * 28):
    return max(1, int(pixels / merge))

CAPS = [("256*28*28", 256 * 28 * 28), ("512*28*28", 512 * 28 * 28),
        ("1280*28*28 (default)", 1280 * 28 * 28), ("4096*28*28", 4096 * 28 * 28)]

print(f"{'cap':<24}{'tokens':>8}" + "".join(f"{t[:13]:>15}" for t in TASKS))
print("-" * 92)
for label, px in CAPS:
    row = [quality(t, px) for t in TASKS]
    print(f"{label:<24}{tokens_for(px):>8,}" + "".join(f"{v:>14.0%} " for v in row))

print("\nRead across the rows: at the SAME cost, different tasks get wildly different value.")
print("At 256 tokens, scene description is already at 93% of its ceiling while document OCR")
print("is at a small fraction of its own. One global max_pixels serves neither well.")

print("\nCost-effectiveness (quality per 1000 tokens) - where the knee is for each task:\n")
print(f"{'task':<20}" + "".join(f"{lbl.split()[0]:>14}" for lbl, _ in CAPS))
print("-" * 78)
for t in TASKS:
    vals = []
    for _, px in CAPS:
        vals.append(quality(t, px) / (tokens_for(px) / 1000))
    print(f"{t:<20}" + "".join(f"{v:>14.2f}" for v in vals))
print("\nHigher is better. Every task peaks at the LOWEST cap - because quality saturates while")
print("cost keeps climbing. The question is never 'what is the best resolution', it is")
print("'what is the lowest resolution that clears my quality bar for THIS task' (Part 4).")

## Part 2 · Caching: the free win, with a VLM-specific catch

Two distinct caches matter, and they save different phases:

| Cache | Saves | Keyed on |
|---|---|---|
| **Image embedding cache** | CPU preprocessing **+ vision encoder** | the image bytes (a hash) |
| **Multimodal prefix cache** | **LLM prefill** over image + text tokens | the resulting token sequence |

**The catch that makes or breaks it:** the prefix cache works on *token IDs*. Image tokens are
placeholders, so two requests only share a prefix if the images produce **byte-identical**
embeddings. Re-encoding the same photo after a re-upload (different JPEG compression, different
EXIF) yields a different hash and misses both caches.

The mitigations are worth knowing because they're cheap:
- hash the **decoded pixel buffer** after resize, not the uploaded file bytes;
- have clients send a stable **image ID** where the application allows it;
- keep the resize deterministic — same interpolation, same target size, every time.

In [ ]:
random.seed(7)

def simulate_caching(n_requests=600, n_unique_images=80, turns_per_image=3,
                     hash_stability=1.0, cache_size=200,
                     preprocess_ms=18.0, vit_ms=9.0, prefill_ms=95.0):
    # Traffic: a set of images, each asked about over several conversational turns.
    reqs = []
    for _ in range(n_requests // turns_per_image):
        img = random.randrange(n_unique_images)
        for turn in range(turns_per_image):
            reqs.append({"img": img, "turn": turn})
    random.shuffle(reqs)

    embed_cache, prefix_cache = {}, {}
    clock = 0
    saved_ms = total_ms = 0
    hits = {"embed": 0, "prefix": 0, "miss": 0}

    for r in reqs:
        clock += 1
        # A hash that is not perfectly stable (re-uploads, re-encodes) misses the cache.
        stable = random.random() < hash_stability
        key = r["img"] if stable else (r["img"], clock)

        full_cost = preprocess_ms + vit_ms + prefill_ms
        cost = full_cost
        if key in embed_cache:
            hits["embed"] += 1
            cost -= (preprocess_ms + vit_ms)              # skip CPU decode AND the ViT
            # a repeated image on a later turn also shares the LLM prefix
            if (key, r["turn"]) in prefix_cache or r["turn"] > 0:
                hits["prefix"] += 1
                cost -= prefill_ms * 0.9
        else:
            hits["miss"] += 1
        embed_cache[key] = clock
        prefix_cache[(key, r["turn"])] = clock
        if len(embed_cache) > cache_size:                  # LRU eviction
            for k in sorted(embed_cache, key=embed_cache.get)[:len(embed_cache) - cache_size]:
                del embed_cache[k]
        total_ms += cost
        saved_ms += full_cost - cost

    n = len(reqs)
    return {"requests": n, "avg_ms": total_ms / n, "baseline_ms": preprocess_ms + vit_ms + prefill_ms,
            "saved_frac": saved_ms / (n * (preprocess_ms + vit_ms + prefill_ms)),
            "embed_hit": hits["embed"] / n, "prefix_hit": hits["prefix"] / n}

print("600 requests over 80 unique images, ~3 turns each (a typical image-chat workload):\n")
print(f"{'hash stability':<20}{'embed hits':>12}{'prefix hits':>13}{'avg ms/req':>13}{'saved':>9}")
print("-" * 70)
for label, stab in [("perfect (1.00)", 1.00), ("good (0.95)", 0.95),
                    ("mediocre (0.80)", 0.80), ("broken (0.30)", 0.30)]:
    s = simulate_caching(hash_stability=stab)
    print(f"{label:<20}{s['embed_hit']:>12.0%}{s['prefix_hit']:>13.0%}"
          f"{s['avg_ms']:>12.0f}ms{s['saved_frac']:>9.0%}")

base = simulate_caching(hash_stability=1.0)
print(f"\nWith a stable hash you cut average request cost from {base['baseline_ms']:.0f}ms to "
      f"{base['avg_ms']:.0f}ms.")
print("With an unstable hash the same code delivers a fraction of that - and nothing in your")
print("dashboards says 'your image hash is unstable'. You just see a low hit rate (logs).")

print("\nCache size sweep at perfect hashing (80 unique images in circulation):\n")
print(f"{'cache entries':>14}{'embed hits':>13}{'avg ms/req':>13}")
print("-" * 42)
for size in (10, 25, 50, 100, 200):
    s = simulate_caching(cache_size=size)
    print(f"{size:>14}{s['embed_hit']:>13.0%}{s['avg_ms']:>12.0f}ms")
print("\nThe knee is at roughly the working-set size. Sizing the cache below it wastes the")
print("feature entirely - the same lesson as KV block capacity in serving-fundamentals.")

## Part 3 · Visual token pruning — the subtle one

The observation behind every pruning method: **most image tokens receive very little attention from
the text tokens that actually matter.** A 1,200-token image might have 200 tokens doing the work.

| Method | When it prunes | How it chooses |
|---|---|---|
| **Pooling / merging** | before the LLM | spatial average of neighbouring patches |
| **FastV** | after layer K of the LLM | attention received, measured at that layer |
| **Similarity merging** | before the LLM | merges near-duplicate patches (sky, background) |
| **Query-aware selection** | before the LLM | keeps patches relevant to the *question* |

The nuance that gets missed: **pruning after layer K only saves the layers after K.** FastV pruning
at layer 2 of 32 saves ~94% of the image-token compute; pruning at layer 16 saves half. But pruning
early means deciding with less information — the accuracy/savings trade is *literally* a depth
choice.

And a second nuance that matters more for serving: **pruning before the KV cache is written saves
KV; pruning after does not.** Many papers report FLOP savings while the KV cost is unchanged.

In [ ]:
def prune_savings(n_image_tokens, n_text_tokens, n_layers, prune_at_layer, keep_frac,
                  prunes_kv=True):
    # Prefill compute is proportional to tokens x layers (ignoring attention's quadratic term,
    # which only makes pruning look better).
    full = (n_image_tokens + n_text_tokens) * n_layers
    before = (n_image_tokens + n_text_tokens) * prune_at_layer
    after = (n_image_tokens * keep_frac + n_text_tokens) * (n_layers - prune_at_layer)
    compute_saving = 1 - (before + after) / full
    kv_tokens = (n_image_tokens * keep_frac + n_text_tokens) if prunes_kv else \
                (n_image_tokens + n_text_tokens)
    kv_saving = 1 - kv_tokens / (n_image_tokens + n_text_tokens)
    return {"compute_saving": compute_saving, "kv_saving": kv_saving}

IMG, TXT, LAYERS = 1222, 16, 28
print(f"Qwen2-VL-7B shape: {IMG} image tokens, {TXT} text tokens, {LAYERS} layers.")
print("Keeping 25% of image tokens, varying WHERE the pruning happens:\n")
print(f"{'prune at layer':>15}{'compute saved':>16}{'KV saved':>11}")
print("-" * 44)
for layer in (0, 2, 4, 8, 16, 24):
    s = prune_savings(IMG, TXT, LAYERS, layer, 0.25)
    print(f"{layer:>15}{s['compute_saving']:>15.0%}{s['kv_saving']:>11.0%}")

print("\nPruning at layer 0 (before the LLM) saves the most compute AND all the KV.")
print("Pruning at layer 16 saves about half the compute - and, if the KV for those tokens")
print("was already written, none of the memory.\n")

print("The same table when pruning does NOT free KV (a common implementation detail):\n")
print(f"{'prune at layer':>15}{'compute saved':>16}{'KV saved':>11}")
print("-" * 44)
for layer in (2, 8, 16):
    s = prune_savings(IMG, TXT, LAYERS, layer, 0.25, prunes_kv=False)
    print(f"{layer:>15}{s['compute_saving']:>15.0%}{s['kv_saving']:>11.0%}")
print("\nIf you are choosing a pruning method for SERVING (not for a FLOPs table in a paper),")
print("ask whether it frees KV. Concurrency is usually the constraint that costs you money")
print("(serving-fundamentals, benchmarking), and a method that halves FLOPs but keeps all the KV does not raise it.")

In [ ]:
# The accuracy/savings trade, and where the knee is.
def pruning_quality(keep_frac, task_sensitivity):
    # Quality degrades slowly at first (redundant patches), then sharply once real signal is cut.
    # task_sensitivity: 1.0 = tolerant (scene description), 3.0 = fragile (OCR).
    return max(0.0, 1 - (1 - keep_frac) ** (1 / task_sensitivity) * 0.55)

SENS = {"scene description": 1.0, "object counting": 1.8, "chart reading": 2.4, "document OCR": 3.2}
KEEPS = [1.0, 0.75, 0.5, 0.35, 0.25, 0.15, 0.10]

print(f"{'keep':>6}{'compute saved':>15}" + "".join(f"{t[:12]:>14}" for t in SENS))
print("-" * 76)
pareto = []
for keep in KEEPS:
    s = prune_savings(IMG, TXT, LAYERS, 2, keep)
    row = {t: pruning_quality(keep, v) for t, v in SENS.items()}
    pareto.append({"keep": keep, "saved": round(s["compute_saving"], 3),
                   "kv_saved": round(s["kv_saving"], 3),
                   **{t: round(v, 3) for t, v in row.items()}})
    print(f"{keep:>6.0%}{s['compute_saving']:>14.0%}" + "".join(f"{row[t]:>13.0%} " for t in SENS))

print("\nDocument OCR degrades fast - it needs the very patches pruning throws away.")
print("Scene description tolerates aggressive pruning almost for free.")
print("\nThe practical consequence, which is the whole point of Part 4:")
print("  the correct pruning ratio is a property of the REQUEST, not of the deployment.")

In [ ]:
JS = r'''
const tasks = Object.keys(data[0]).filter(k => !["keep","saved","kv_saved"].includes(k));
const M = {top: 22, right: 150, bottom: 46, left: 60};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleLinear().domain([0, d3.max(data,d=>d.saved)]).range([0,iw]);
const y = d3.scaleLinear().domain([0.4, 1.02]).range([ih,0]);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x).tickFormat(d3.format(".0%")));
svg.append("g").call(d3.axisLeft(y).tickFormat(d3.format(".0%")));
svg.append("text").attr("x",iw/2).attr("y",ih+38).attr("text-anchor","middle")
   .style("font-size","12px").text("prefill compute saved by pruning →");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-44)
   .attr("text-anchor","middle").style("font-size","12px").text("relative quality");
const color = d3.scaleOrdinal().domain(tasks).range(["#2e7d32","#1976d2","#ef6c00","#c62828"]);
const line = d3.line().x(d=>x(d.saved)).y(d=>y(d.q));
tasks.forEach(t => {
  const pts = data.map(d => ({saved: d.saved, q: d[t], keep: d.keep}));
  svg.append("path").datum(pts).attr("fill","none").attr("stroke",color(t))
     .attr("stroke-width",2).attr("d",line);
  svg.selectAll("p"+t).data(pts).join("circle")
     .attr("cx",d=>x(d.saved)).attr("cy",d=>y(d.q)).attr("r",3).attr("fill",color(t))
     .append("title").text(d=>`${t}: keep ${d3.format(".0%")(d.keep)} → ${d3.format(".0%")(d.q)} quality, ${d3.format(".0%")(d.saved)} saved`);
});
tasks.forEach((t,i)=>{
  svg.append("rect").attr("x",iw+12).attr("y",i*20).attr("width",11).attr("height",11)
     .attr("rx",2).attr("fill",color(t));
  svg.append("text").attr("x",iw+28).attr("y",i*20+10).style("font-size","10.5px").text(t);
});
svg.append("line").attr("x1",0).attr("x2",iw).attr("y1",y(0.95)).attr("y2",y(0.95))
   .attr("stroke","#888").attr("stroke-dasharray","4 3");
svg.append("text").attr("x",4).attr("y",y(0.95)-4).style("font-size","10px").style("fill","#888")
   .text("95% quality bar");
'''
show_d3(JS, pareto, height=360)

**Where each curve crosses the 95% bar is that task's safe pruning budget** — and they cross at
very different places. A single global pruning ratio either wastes money on easy requests or breaks
hard ones.

## Part 4 · Adaptive resolution & pruning: route by task

This is the technique that makes the previous two parts practical. Classify the request cheaply,
then pick resolution and pruning per request:

```
request ──► cheap classifier ──► "document/chart?"  ──yes──► high res, no pruning
                                        │
                                        no
                                        ▼
                                  low res, aggressive pruning
```

The classifier can be almost anything: the text prompt alone ("read the text in this", "what does
the chart say") is a strong signal, and it costs nothing compared to the image pipeline. The risk is
**routing errors**, so the cost model has to include them.

In [ ]:
NEEDS_HIGH = {"chart reading", "document OCR"}

def evaluate_policy(mix, route_fn, trials=600, high=1222, low=256):
    # Track per-task quality as well as the average - the average hides misrouting.
    cost = 0.0
    per_task = {t: [] for t in mix}
    for _ in range(trials):
        for task, share in mix.items():
            tokens = high if route_fn(task) else low
            cost += share * tokens / trials
            per_task[task].append(quality(task, tokens * 28 * 28))
    per_task_avg = {t: statistics.mean(v) for t, v in per_task.items()}
    avg = sum(per_task_avg[t] * s for t, s in mix.items())
    return {"avg_tokens": cost, "quality": avg, "per_task": per_task_avg,
            "worst_task": min(per_task_avg, key=per_task_avg.get),
            "worst_quality": min(per_task_avg.values())}

random.seed(3)
MIX = {"scene description": 0.45, "object counting": 0.25,
       "chart reading": 0.20, "document OCR": 0.10}

rows = []
rows.append(("always high res", evaluate_policy(MIX, lambda t: True)))
rows.append(("always low res", evaluate_policy(MIX, lambda t: False)))
for acc in (1.0, 0.92, 0.75):
    rows.append((f"adaptive (classifier {acc:.0%})",
                 evaluate_policy(MIX, lambda t, a=acc:
                                 (t in NEEDS_HIGH) if random.random() < a
                                 else (t not in NEEDS_HIGH))))

print(f"Traffic mix: {', '.join(f'{k} {v:.0%}' for k, v in MIX.items())}\n")
print(f"{'policy':<30}{'avg tokens':>11}{'avg quality':>13}{'WORST task':>13}   worst is")
print("-" * 88)
for label, r in rows:
    print(f"{label:<30}{r['avg_tokens']:>11,.0f}{r['quality']:>13.1%}"
          f"{r['worst_quality']:>13.1%}   {r['worst_task']}")

print("\nRead the AVERAGE column and the WORST column against each other - they disagree,")
print("and the disagreement is the entire lesson:")
print("  - average quality RISES as the classifier gets worse, because misroutes send more")
print("    traffic to high resolution. The aggregate looks fine, or even improved.")
print("  - the WORST task collapses, because that is where documents got sent to low res.")
print("\nSo a bad classifier does not show up as 'quality went down'. It shows up as one task")
print("silently failing while your dashboard average drifts UP - which is exactly the")
print("aggregate-vs-per-case failure fine-tune-to-production's quality gate exists to catch.")
print("Gate adaptive routing per task, or do not ship it.")

## Part 5 · Encoder/LLM disaggregation

The vision encoder and the LLM are different workloads sharing a GPU — the same conflict [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)
described for prefill vs decode, one level up:

| | Vision encoder | LLM decode |
|---|---|---|
| Bound by | compute | memory bandwidth |
| Batching | large batches, uniform shapes | continuous, variable |
| Duration | milliseconds | seconds |
| Scales with | images/second | concurrent conversations |

Colocated, a burst of image uploads stalls everyone's token stream. Disaggregated, you scale
"images/second" and "conversations" independently — and can put the ViT on cheaper hardware, since
it's a small compute-bound model that doesn't need 80GB of HBM.

**The cost is a network hop for the embeddings**, and that is small: an embedding tensor is
`tokens × hidden × dtype`, which is far smaller than a KV cache transfer ([Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)).

In [ ]:
def disagg_transfer(img_tokens, hidden=3584, bytes_per=2, link_gbs=25):
    payload = img_tokens * hidden * bytes_per
    return {"payload_mb": payload / 1e6, "transfer_ms": payload / (link_gbs * 1e9) * 1000}

print("Embedding transfer cost for encoder/LLM disaggregation (Qwen2-VL hidden=3584):\n")
print(f"{'image tokens':>14}{'payload':>11}{'transfer @25Gb':>17}{'vs ViT compute':>17}")
print("-" * 62)
for toks in (256, 576, 1222, 2880, 7000):
    d = disagg_transfer(toks)
    vit_ms = (2 * 0.68e9 * toks) / (312e12 * 0.6) * 1000
    print(f"{toks:>14,}{d['payload_mb']:>9.1f}MB{d['transfer_ms']:>15.1f}ms"
          f"{d['transfer_ms']/vit_ms:>16.2f}x")

print("\nThe transfer is comparable to the ViT compute itself at typical token counts, so")
print("disaggregation is NOT free - it pays off when the interference it removes costs more")
print("than the hop it adds. That is true at scale (many concurrent streams being interrupted")
print("by image bursts) and false for a single-GPU deployment, exactly like the distributed-serving notebook's conclusion.")

## Part 6 · Quantization: the vision tower is more fragile

A nuance with real consequences: **the ViT tolerates quantization worse than the LLM does.** It's
small (0.3–0.7B), so quantizing it saves little memory, and its outputs feed *every* downstream
token — an error there propagates through the whole response rather than affecting one token.

The standard practice follows directly:

```
LLM   (7B+, most of the weights)  → quantize aggressively (AWQ/GPTQ int4, FP8)
ViT   (0.3-0.7B, small)           → keep at fp16/bf16
projector (tiny)                  → keep at fp16/bf16
```

In [ ]:
def vlm_memory(vit_b, llm_b, vit_bytes, llm_bytes):
    return {"vit_gb": vit_b * vit_bytes / 1e9, "llm_gb": llm_b * llm_bytes / 1e9,
            "total_gb": (vit_b * vit_bytes + llm_b * llm_bytes) / 1e9}

VIT, LLM = 0.68e9, 7e9
print("Qwen2-VL-7B weight memory under different quantization strategies:\n")
print(f"{'strategy':<34}{'ViT':>8}{'LLM':>8}{'total':>9}{'vs fp16':>9}")
print("-" * 70)
base = vlm_memory(VIT, LLM, 2, 2)["total_gb"]
for label, vb, lb in [("everything fp16", 2, 2),
                      ("LLM int4, ViT fp16 (recommended)", 2, 0.5),
                      ("everything int4", 0.5, 0.5),
                      ("LLM fp8, ViT fp16", 2, 1)]:
    m = vlm_memory(VIT, LLM, vb, lb)
    print(f"{label:<34}{m['vit_gb']:>6.2f}GB{m['llm_gb']:>6.2f}GB{m['total_gb']:>7.2f}GB"
          f"{m['total_gb']/base:>8.0%}")

rec = vlm_memory(VIT, LLM, 2, 0.5)["total_gb"]
allq = vlm_memory(VIT, LLM, 0.5, 0.5)["total_gb"]
print(f"\nQuantizing the ViT as well saves only {rec - allq:.2f} GB more "
      f"({(rec-allq)/rec:.1%} of the already-quantized total)")
print("- and puts the component every downstream token depends on at risk.")
print("\nThat is the whole argument: the ViT is a small fraction of the memory and a large")
print("fraction of the quality risk. Quantize the LLM, leave the vision tower alone,")
print("and re-run the quality gate from fine-tune-to-production at YOUR resolution afterwards.")

## Part 7 · Video: the same math, one dimension worse

Video is images × frames, and the token count explodes accordingly. The whole game is **frame
sampling**: how few frames can you keep and still answer the question?

In [ ]:
def video_tokens(duration_s, fps_sampled, tokens_per_frame, temporal_merge=2):
    frames = max(1, int(duration_s * fps_sampled))
    # many VLMs merge adjacent frames temporally (Qwen2-VL merges pairs)
    return frames * tokens_per_frame // temporal_merge, frames

print("A 60-second clip at 256 tokens/frame (Qwen2-VL-style temporal merging of 2):\n")
print(f"{'sampling':<22}{'frames':>8}{'tokens':>10}{'KV @ 4 kv-heads':>18}   feasible?")
print("-" * 74)
kv_per_token = 2 * 28 * 4 * 128 * 2
for label, fps in [("1 frame / 4s (0.25)", 0.25), ("1 fps", 1.0),
                   ("2 fps", 2.0), ("8 fps", 8.0), ("24 fps (native)", 24.0)]:
    toks, frames = video_tokens(60, fps, 256)
    kv_gb = kv_per_token * toks / 1e9
    ok = "yes" if toks < 32768 else ("tight" if toks < 131072 else "NO - exceeds context")
    print(f"{label:<22}{frames:>8}{toks:>10,}{kv_gb:>16.2f}GB   {ok}")

print("\nNative frame rate is not a serving option; it is not even a context-length option.")
print("Every production video VLM samples aggressively, and the sampling strategy is where")
print("the quality lives:")
print("  - uniform sampling         : simple, misses short events")
print("  - keyframe / scene-change  : better coverage per token, needs a cheap detector")
print("  - query-aware sampling     : sample densely where the question points")
print("\nThe serving lesson is the same as Part 4: the right sampling rate is a property of the")
print("REQUEST. 'Count how many times the door opens' needs frames; 'describe this room' does not.")

## Part 8 · How these compose

Applying [The Optimization Stack](./The_Optimization_Stack.ipynb)'s method to the VLM toolbox — because these
interact at least as strongly as the text-only ones:

In [ ]:
INTERACTIONS = [
    ("resolution cap", "token pruning", 0.55,
     "both remove image tokens - the second one has far less left to remove"),
    ("resolution cap", "embedding cache", 1.00,
     "independent: one shrinks the work, the other skips repeats of it"),
    ("embedding cache", "prefix cache", 1.25,
     "synergistic: a cached embedding produces identical tokens, so the prefix hits too"),
    ("token pruning", "prefix cache", 0.70,
     "ANTAGONISTIC: query-aware pruning makes the token sequence depend on the QUESTION,"
     " so two turns about the same image no longer share a prefix"),
    ("token pruning", "fp8 KV cache", 0.80,
     "both shrink KV; whichever runs second saves less"),
    ("disaggregation", "embedding cache", 1.15,
     "the cache lives naturally in the encoder tier and serves all LLM replicas"),
    ("adaptive resolution", "embedding cache", 0.85,
     "routing the same image at different resolutions fragments the cache key"),
    ("LLM int4", "ViT fp16", 1.00,
     "independent components; this is the recommended pairing (Part 6)"),
]
print(f"{'A':<22}{'B':<20}{'synergy':>9}   why")
print("-" * 100)
for a, b, syn, why in sorted(INTERACTIONS, key=lambda r: r[2]):
    flag = "🔴" if syn < 0.9 else ("🟢" if syn > 1.1 else "⚪")
    print(f"{a:<22}{b:<20}{syn:>9.2f} {flag} {why}")

print("\nThe row worth reading twice is `token pruning x prefix cache`.")
print("Query-aware pruning is often presented as strictly better than static pruning - it is")
print("more accurate per token kept. But it makes the image's token sequence depend on the")
print("QUESTION, which destroys prefix reuse across turns about the same image. For a")
print("multi-turn image chat (RAG & agents's shape), the caching you lose can cost more than the")
print("pruning gains. Static, query-independent pruning is cache-friendly; query-aware is not.")
print("\nThat trade does not appear in any pruning paper, because papers measure single-turn")
print("accuracy per FLOP, not multi-turn cost per conversation.")

## Part 9 · Recipes

Derived from the parts above, per workload shape:

| Workload | Resolution | Pruning | Caching | Notes |
|---|---|---|---|---|
| **Consumer photo chat** | low cap (256–512) | aggressive, static | embedding + prefix | quality saturates early (Part 1) |
| **Document / OCR** | high cap, no pruning | none | embedding cache | pruning removes exactly the signal (Part 3) |
| **Chart / diagram QA** | medium-high | light, static | embedding + prefix | verify at your resolution |
| **Mixed traffic** | adaptive routing | per-route | embedding cache | measure the classifier first (Part 4) |
| **Video understanding** | frame sampling first | temporal merging | keyframe cache | sampling dominates everything else (Part 7) |
| **Multi-image compare** | low-medium cap | static | embedding cache | watch `--limit-mm-per-prompt` ([VLM Serving: The Token Explosion](./VLM_Serving_Token_Explosion.ipynb)) |

### The VLM serving checklist

- [ ] `max_pixels` / `max_num` **set deliberately from the task**, not left at the default ([VLM Serving: The Token Explosion](./VLM_Serving_Token_Explosion.ipynb))
- [ ] `--limit-mm-per-prompt` set — one request must not be able to eat the KV pool
- [ ] Image hashing done on the **decoded, resized buffer**, and hit rate monitored (Part 2)
- [ ] Preprocessing worker count checked against the GPU stage ([VLM Serving: The Token Explosion](./VLM_Serving_Token_Explosion.ipynb) Part 3)
- [ ] Pruning method chosen knowing **whether it frees KV** (Part 3)
- [ ] If query-aware pruning: accept that multi-turn prefix reuse is gone (Part 8)
- [ ] ViT left at fp16 even when the LLM is int4 (Part 6)
- [ ] Quality gate ([From Fine-Tune to Production](./From_FineTune_To_Production.ipynb)) re-run **at your production resolution**, not at 336px
- [ ] Video: sampling rate decided per request type before anything else

## Recap

1. **Resolution is the master dial**, and quality saturates — usually well before the default cap.
2. **Caching is the free win**, but it hinges on a stable image hash; hash the decoded buffer.
3. **Pruning's value depends on *where* it happens** — before the LLM saves compute *and* KV; late
   pruning may save neither in a way that raises concurrency.
4. **The right resolution and pruning ratio are properties of the request**, so routing beats any
   fixed global setting — provided the classifier is actually good.
5. **The vision tower is small and fragile**: quantize the LLM, leave the ViT alone.
6. **Query-aware pruning silently destroys prefix caching** for multi-turn image chat. That
   interaction is invisible in single-turn benchmarks and expensive in production.

### Further reading
- [FastV](https://arxiv.org/abs/2403.06764) — pruning by attention at layer K · [LLaVA-PruMerge](https://arxiv.org/abs/2403.15388)
- [Qwen2-VL](https://arxiv.org/abs/2409.12191) (dynamic resolution + temporal merging) · [InternVL](https://arxiv.org/abs/2312.14238)
- [vLLM multimodal docs](https://docs.vllm.ai/en/latest/features/multimodal_inputs.html)
- Foundations: [42 VLM mechanics](./VLM_Serving_Token_Explosion.ipynb) · [40 The Optimization Stack](./The_Optimization_Stack.ipynb) · [21 quality gates](./From_FineTune_To_Production.ipynb)

🏁 [Back to the learning path](README.md)